In [8]:
import os
import numpy as np
import pandas as pd
import torch

import mlflow
import mlflow.transformers
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve, auc
)
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)
import seaborn as sns

# Ensure results folder exists
os.makedirs("../models/distilbert_fake_news", exist_ok=True)

In [9]:
# ==============================
# Step 1. Load tokenized dataset
# ==============================
dataset_path = "../data/processed/tokenized"
dataset = load_from_disk(dataset_path)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['label', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 35918
    })
    validation: Dataset({
        features: ['label', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 4490
    })
    test: Dataset({
        features: ['label', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 4490
    })
})


In [10]:
# ==============================
# Step 2. Load tokenizer and model
# ==============================
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# ==============================
# Step 3. Define metrics function
# ==============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
# ==============================
# Step 4. TrainingArguments
# ==============================
training_args = TrainingArguments(
    output_dir="../models/distilbert_fake_news",
    evaluation_strategy="steps",
    save_strategy="steps",      
    eval_steps=200,       
    save_steps=200,         
    learning_rate=3e-5,
    per_device_train_batch_size=4,      
    per_device_eval_batch_size=4,
    num_train_epochs=2,                 
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="logs",
    logging_strategy="steps",
    logging_steps=50,                   
    report_to="none",
    dataloader_num_workers=0,           
    gradient_accumulation_steps=2,      
    fp16=False,                         
    disable_tqdm=False,
    log_level="error"
)


In [15]:
# ==============================
# Step 5. Trainer initialization
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

C:\Users\ajsal\AppData\Local\Temp\ipykernel_15856\2892208812.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
# ==============================
# Step 6. MLflow setup
# ==============================
mlflow.set_experiment("FakeNews_DistilBERT")

with mlflow.start_run(run_name="distilbert_run"):
    mlflow.log_params({
        "model_name": model_name,
        "epochs": training_args.num_train_epochs,
        "lr": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size
    })

    trainer.train()

    eval_metrics = trainer.evaluate(dataset["test"])
    mlflow.log_metrics(eval_metrics)

    trainer.save_model("../models/distilbert_fake_news")
    mlflow.log_artifacts("../models/distilbert_fake_news")

    print("✅ Training complete. Metrics:", eval_metrics)

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ==============================
# Step 7. Evaluate & visualize
# ==============================
predictions = trainer.predict(dataset["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Compute confusion matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Fake News Detection")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(labels, predictions.predictions[:, 1])
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (area = {roc_auc:.2f})")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

print(f"Test Accuracy: {accuracy_score(labels, preds):.4f}")

In [ ]:
# ==============================
# Step 8. Explainability (SHAP)
# ==============================
sample_texts = [
    "Breaking news: Scientists discovered water on Mars!",
    "The president was seen on Mars, confirms local news source.",
    "NASA confirms successful moon mission."
]

explainer = shap.Explainer(model, tokenizer)
shap_values = explainer(sample_texts)
shap.plots.text(shap_values)